In [ ]:
"""
Comparing geocoding results from different geocoders
"""
import datetime
import os
import math

import pandas as pd
import numpy as np
import geopandas as gpd
from IPython.display import display

from matplotlib.colors import ListedColormap
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import pyplot as plt
import seaborn as sbn

from tqdm import tqdm

import shapely
import contextily as ctx
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Here, Nominatim, Bing, Photon, MapBox, TomTom, GoogleV3, ArcGIS

from geopy_nominatim_wrapper import NominatimWrapper
from geopy_bestaddress import BestAddress
from geopy_pelias import Pelias
from geopy_bepelias import BePelias
from geopy_ngi import NGI
from geopy_irisnet import IrisNet

from credentials import (here_api_key,
                         bing_api_key,
                         mapbox_api_key,
                         tomtom_api_key,
                         google_api_key,
                         arcgis_username, arcgis_password,
                         best_client_id, best_client_secret, best_hostname, best_prefix, belgov_trace_id)

tqdm.pandas()

fig_path = "output/geocoding/figs"  # pylint: disable=invalid-name
data_dir = "data/geocoding"         # pylint: disable=invalid-name
output_dir = "output/geocoding/"    # pylint: disable=invalid-name

In [ ]:
# dataset = "rrn_1000"
dataset = "kbo_1000"
# dataset = "kbo_bru_1000"
# dataset = "kbo_bru_nl_1000"
# dataset = "best_1000"
# dataset = "best2_1000"
# dataset = "rep_1000"
# dataset = "resto_1000"

dataset = "kbo_10000"
# dataset = "rrn_10000"
# dataset = "rep_10000"
# dataset = "best_10000"

In [ ]:
crs = 'epsg:3857'       # pylint: disable=invalid-name
osm_crs = 'epsg:4326'   # pylint: disable=invalid-name

def_figsize = (6, 4)

In [ ]:
os.makedirs(f"{output_dir}/geocoded_data/", exist_ok=True)
os.makedirs(f"{output_dir}/reports/", exist_ok=True)
os.makedirs(f"{output_dir}/no_res/", exist_ok=True)
os.makedirs(fig_path, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)


In [ ]:


def add_basemap(ax, zoom='auto'):
    """
    Add a basemap on a plot. Tries first default (Stamen) basemap. If errors, tries OpenStreetMap.Mapnik

    Parameters
    ----------
    ax: matplotlib axes

    Returns
    -------
        None
    """

    try:
        ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik, zoom=zoom)
    except ValueError as e:
        print("Value error...")
        print(e)
    except NameError:
        print("contextily not installed")


In [ ]:


def set_optimal_limits(ax, df):
    """
    Adapt xlim/ylim to a GeoDataFrame point plot to avoid plot to be too wide when
    points are horizontally aligned, and to narrow when points are vertically aligned

    Usage:

    ax = df.plot()
    set_optimal_limits(ax, df)

    Parameters
    ----------
    ax: AxesSubplot
       plot to resize
    df: GeoDataFrame
       data to be plotted

    Returns
    -------
        None
    """

    plot_ratio = 1.5    # Optimal ratio between "one horizontal degree" and "one vertical degree". It depends of the CRS.
                        # For "polar" CRS, it may also depend of the place on the globe   # noqa: E116
    minimal_width = 200

    margins = 1.1  # Avoid having dots on edges of the plot

    # Compute dimension of the data
    xmin, ymin, xmax, ymax = df.total_bounds
    height = ymax - ymin
    width = xmax - xmin

    opt_height = max(height, width / plot_ratio, minimal_width / plot_ratio)
    opt_width = max(width, height*plot_ratio, minimal_width)

    # If plot is too narrow, increase xmin. If plot is too wide, increase ylim

    if opt_height > height:
        ymid = (ymax + ymin) / 2
        mid_height = opt_height * margins / 2
        ax.set_ylim(ymid - mid_height, ymid + mid_height)
    if opt_width > width:
        xmid = (xmax + xmin) / 2
        mid_width = opt_width * margins / 2
        ax.set_xlim(xmid - mid_width, xmid + mid_width)


In [ ]:


def pdf_savefig():
    """  Save the current figure to the PDF file. If the PDF file is closed, catch the exception and print a message. """
    try:
        pdf.savefig(bbox_inches='tight')
    except AttributeError:
        print("PDF probably closed")


# Prepare geocoders

In [ ]:


def get_precision(record):
    """ Extract precision level"""

    precision_field = {
        "nominatim": "class",
        "nominatim_loc": "class",
        "nominatim_wrapper": ("metadata", "placeRank"),
        "here": "MatchLevel",
        "photon": ("properties", "type"),
        "bing": "entityType",
        "mapbox": ("properties", "accuracy"),
        "tomtom": "type",
        "trillium": "match_level",
        "google": ("geometry", "location_type"),
        "bestaddress": "precision",
        "lpost_bestaddress": "precision",
        "be_pelias": "precision",
        "ngi": "precision",
        "ngi_struct": "precision",
        "ngi_prd": "precision",
        "ngi_prd_struct": "precision",
        "irisnet": "precision",
        "irisnet_struct": "precision",
     }

    label_mapping = {
        "nominatim": {"building": "building",
                      "amenity":  "building",
                      "shop":     "building",
                      "place":    "building",
                      "highway":  "street"},
        "nominatim_loc": {"building": "building",
                          "amenity":  "building",
                          "shop":     "building",
                          "place":    "building",
                          "highway":  "street"},
        "nominatim_wrapper": {"30": "building",
                              "27": "street",
                              "26": "street",
                              "25": "city",
                              "22": "city",
                              "21": "city",
                              "20": "city",
                              "19": "city",
                              "18": "city",
                              "16": "city",
                              "14": "city"},

        "here": {"houseNumber": "building",
                 "street": "street",
                 "city": "city",
                 "postalCode": "city"},
        "photon": {
            "house": "building",
            "locality": "city",
            "district": "city",
            "city": "city",
            "street": "street"},
        "bing": {"Address": "building",
                 "RoadBlock": "street",
                 "PopulatedPlace": "city",
                 "Postcode1": "city",
                 "CountryRegion": "country"},
        "mapbox": {"rooftop": "building",
                   "point": "building",
                   "interpolated": "building",
                   "street": "street"},
        "tomtom": {"Point Address": "building",
                   "Street": "street",
                   "Address Range": "street",
                   "Cross Street": "street",
                   "Geography": "city"},
        "trillium": {0: "building",
                     2: "city",
                     3: "street"},
        "google": {
            "ROOFTOP": "building",
            "RANGE_INTERPOLATED": "building",
            "GEOMETRIC_CENTER": "street",
            "APPROXIMATE": "city"},
        "bestaddress": {
            "address": "building",
            "address_nocityname": "building",
            "~address": "building",
            "~address_nocityname": "building",
            "address_cleanstr": "building",
            "~address_cleanstr": "building",
            "~address_cleanhn": "building",
            "street": "street",
            "~street": "street",
            "~clean_street": "street",
            "street2": "country",
            "~street2": "country",
            "country": "country",
            "city": "city",
            "city_zip": "city",
            "city_postalname": "city",
            "city_municname": "city",
            "address_00": "country",
            "address_nocityname_00": "country",
            "~address_00": "country",
            "~address_nocityname_00": "country",
            "street_00": "country",
            "~street_00": "country",
            "~address_cleanhn_00": "country",
            "~clean_street_00": "country",
            "address_cleanstr_00": "country",
            "~address_cleanstr_00": "country",
            "street2_cleanstr": "country",
            "~street2_cleanstr": "country"},

        "lpost_bestaddress": {},

        "be_pelias": {
            "address":              "building",
            "address_00":           "country",
            "address_streetcenter": "street",
            "address_interpol":     "building",
            "street_interpol":      "building",
            "street_00":            "country",
            "address_postcodecenter": "city",
            "street_postcodecenter": "city"},
        "ngi": {},
        "ngi_struct": {},
        "ngi_prd": {},
        "ngi_prd_struct": {},
        "irisnet": {},
        "irisnet_struct": {}
    }

    if record.geocoder.startswith("pelias"):
        feature = record.location.raw
        feat_prop = feature["properties"]
        if feat_prop["layer"] == "address":
            if feature["geometry"]["coordinates"] == [0, 0]:
                return "country", "address_00"
            if feat_prop["match_type"] == "interpolated":
                if "/streetname/" in feat_prop["id"].lower() or "/straatnaam/" in feat_prop["id"].lower():
                    return "street", "street_interpol"
                return "address_interpol2"  # Should not occur?
            if feat_prop["match_type"] == "exact" or feat_prop["accuracy"] == "point":
                return "building", "address"

        if feat_prop["layer"] == "street":
            if feature["geometry"]["coordinates"] == [0, 0]:
                return "country", "street_00",
            return "street", "street"

        if feat_prop["layer"] in ("city", "locality", "postalcode", "localadmin", "neighbourhood"):
            return "city", "city"

        if feat_prop["layer"] in ("region", "macroregion", "county"):
            return "country", "country"

    if "arcgis" in record.geocoder:
        attr = record.location.raw["attributes"]
        if "AddNum" in attr and len(attr["AddNum"]) > 0:
            return "building"
        if "StName" in attr and len(attr["StName"]) > 0:
            return "street"
        if "City" in attr and len(attr["City"]) > 0:
            return "city"

        return "[UNKNOWN]"

    try:
        geocoder = record.geocoder
        if geocoder.startswith("bestaddress"):
            geocoder = "bestaddress"

        if geocoder.startswith("be_pelias"):
            geocoder = "be_pelias"
        f = precision_field[geocoder]
        mapper = label_mapping[geocoder]

        if isinstance(f, str):
            raw_label = record.location.raw[f]
        else:
            raw_label = record.location.raw[f[0]][f[1]]

        if raw_label in mapper:
            return mapper[raw_label], raw_label
        if raw_label in ["building", "street", "city", "country"]:
            return raw_label, raw_label

        return f"[UNKNOWN - {raw_label}]", raw_label

    except KeyError:
        if record.geocoder == "mapbox":
            return "city", "city"
        return "[UNKNOWN]", "[UNKNOWN]"



In [ ]:

geocoders = {}

In [ ]:
geocoders['ngi']        = NGI(user_agent="smalsresearch", domain="uat.ngi.be", scheme="https", prefix="/geosearchengine", timeout=1000)    # noqa: E221
geocoders['ngi_struct'] = NGI(user_agent="smalsresearch", domain="uat.ngi.be", scheme="https", prefix="/geosearchengine", timeout=1000)    # noqa: E221

In [ ]:
geocoders['ngi_prd']        = NGI(user_agent="smalsresearch", domain="geosearch.ngi.be", scheme="https", timeout=1000)    # noqa: E221
geocoders['ngi_prd_struct'] = NGI(user_agent="smalsresearch", domain="geosearch.ngi.be", scheme="https", timeout=1000)    # noqa: E221

In [ ]:
# geocoders['ngi_prd'].verbose=True
# geocoders['ngi_prd'].geocode("Avenue Fonsny 20, 1060 Bruxelles", exactly_one=False)

In [ ]:
geocoders['irisnet']        = IrisNet("geoservices.irisnetlab.be", user_agent="smalsresearch", timeout=5, language="NL" if dataset == "kbo_bru_nl_1000" else "FR")  # noqa: E221
geocoders['irisnet_struct'] = IrisNet("geoservices.irisnetlab.be", user_agent="smalsresearch", timeout=5, language="NL" if dataset == "kbo_bru_nl_1000" else "FR")  # noqa: E221

In [ ]:
# geocoders['irisnet'].verbose=True
# geocoders['irisnet'].geocode({"street": "Avenue Fonsny", "housenumber": "22", "postcode": 1060, "city": "Bruxelles"}, exactly_one=False)
# geocoders['irisnet'].geocode({"street": "Av des Sept-Bonniers", "housenumber": "", "postcode": 1190, "city": "Forest"}, exactly_one=False)

In [ ]:
geocoders['ngi'].geocode({"street": "Chaussée de Tervueren", "housenumber": "59", "postcode": 1160, "city": "Bruxelles"}, exactly_one=False)

In [ ]:
geocoders['nominatim'] = Nominatim(user_agent="smalsresearch", domain="172.27.0.64:8080", scheme="http", timeout=1000)
# geocoders['nominatim'] = Nominatim(user_agent="smalsresearch")
# geocoders["nominatim"].geocode("Rue Jean-Baptiste Vannypen, 1160 Bruxelles", addressdetails=True, namedetails=True ).raw

In [ ]:
geocoders['nominatim_wrapper'] = NominatimWrapper(user_agent="smalsresearch", scheme="http", timeout=1000, domain="172.27.0.64:5000")

In [ ]:
with_token = False
geocoders['bestaddress_simple'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
                                              client_id=best_client_id, client_secret=best_client_secret,
                                              domain=best_hostname,
                                              prefix=best_prefix,
                                              belgov_trace_id=belgov_trace_id,
                                              verbose=False,
                                              with_token=with_token,
                                              mode="simple")
geocoders['bestaddress_simple_fz'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
                                                 client_id=best_client_id, client_secret=best_client_secret,
                                                 domain=best_hostname,
                                                 prefix=best_prefix,
                                                 belgov_trace_id=belgov_trace_id,
                                                 verbose=False,
                                                 with_token=with_token,
                                                 mode="simple_fuzzy")
geocoders['bestaddress_basic'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
                                             client_id=best_client_id, client_secret=best_client_secret,
                                             domain=best_hostname,
                                             prefix=best_prefix,
                                             belgov_trace_id=belgov_trace_id,
                                             verbose=False,
                                             with_token=with_token,
                                             mode="basic")
geocoders['bestaddress_basic_fz'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
                                                client_id=best_client_id, client_secret=best_client_secret,
                                                domain=best_hostname,
                                                prefix=best_prefix,
                                                belgov_trace_id=belgov_trace_id,
                                                verbose=False,
                                                with_token=with_token,
                                                mode="basic_fuzzy")
geocoders['bestaddress_adv'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
                                           client_id=best_client_id, client_secret=best_client_secret,
                                           domain=best_hostname,
                                           prefix=best_prefix,
                                           belgov_trace_id=belgov_trace_id,
                                           verbose=False,
                                           with_token=with_token,
                                           mode="advanced")
geocoders['bestaddress_adv_chk'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
                                               client_id=best_client_id, client_secret=best_client_secret,
                                               domain=best_hostname,
                                               prefix=best_prefix,
                                               belgov_trace_id=belgov_trace_id,
                                               verbose=False,
                                               with_token=with_token,
                                               mode="advanced",
                                               check_result=True)

In [ ]:
# geocoders['bestaddress_fuzzy'] = BestAddress(user_agent="smalsresearch", scheme="https", timeout=1000,
#                                        client_id=best_client_id, client_secret=best_client_secret,
#                                        domain=best_hostname,
#                                         prefix=best_prefix,
#                                        belgov_trace_id = belgov_trace_id,
#                                        verbose=False,
#                                        fuzzy=True
#                                       )

In [ ]:
# geocoders['bestaddress_adv'].geocode({"street": "Sous le Pont, test", "housenumber": "6", "postcode": "6800", "city": "Libramont-Chevigny"}, exactly_one=False)
# geocoders['bestaddress_adv'].geocode({"street": "Avenue Fonsny", "housenumber": "6", "postcode": "1060", "city": "Libramont-Chevigny"}, exactly_one=False, check_result=True)


In [ ]:
# from geopy.geocoders import Pelias
geocoders['pelias_unstruct']     = Pelias(user_agent="smalsresearch", domain="172.27.0.64:4000", scheme="http", timeout=1000)  # noqa: E221
geocoders['pelias_struct']       = Pelias(user_agent="smalsresearch", domain="172.27.0.64:4000", scheme="http", timeout=1000)  # noqa: E221
geocoders['pelias_struct_noloc'] = Pelias(user_agent="smalsresearch", domain="172.27.0.64:4000", scheme="http", timeout=1000, with_localities=False)


In [ ]:
# from geopy.geocoders import Pelias
# For peliase installed through bePelias procedure
geocoders['be_pelias_basic_unstruct']     = Pelias(user_agent="smalsresearch", domain="172.27.0.64:4000", scheme="http", timeout=1000)  # noqa: E221
geocoders['be_pelias_basic_struct']       = Pelias(user_agent="smalsresearch", domain="172.27.0.64:4000", scheme="http", timeout=1000)  # noqa: E221
geocoders['be_pelias_basic_struct_noloc'] = Pelias(user_agent="smalsresearch", domain="172.27.0.64:4000", scheme="http", timeout=1000, with_localities=False)


In [ ]:
# geocoders['be_pelias'].geocode({"street": "Rue des Vierges", "housenumber": "25", "postcode": "9600", "city": "renaix"}).raw

In [ ]:
geocoders['be_pelias_basic'] = BePelias(user_agent="smalsresearch", domain="172.27.0.64:4001", scheme="http", timeout=1000,
                                        mode="basic")

In [ ]:
geocoders['be_pelias_simple'] = BePelias(user_agent="smalsresearch", domain="172.27.0.64:4001", scheme="http", timeout=1000,
                                         mode="simple")


In [ ]:
geocoders['be_pelias'] = BePelias(user_agent="smalsresearch", domain="172.27.0.64:4001", scheme="http", timeout=1000,
                                  mode="advanced")


In [ ]:
geocoders['be_pelias_struct'] = BePelias(user_agent="smalsresearch", domain="172.27.0.64:4001", scheme="http", timeout=1000,
                                         mode="advanced")


In [ ]:
geocoders['be_pelias_new_struct'] = BePelias(user_agent="smalsresearch", domain="172.27.0.64:4001", scheme="http", timeout=1000,
                                             mode="advanced")

In [ ]:
geocoders['here'] = Here(apikey=here_api_key)
geocoders['bing'] = Bing(api_key=bing_api_key)
geocoders['photon'] = Photon(domain="127.0.0.1:2322", scheme="http")
geocoders['mapbox'] = MapBox(api_key=mapbox_api_key)
geocoders['tomtom'] = TomTom(api_key=tomtom_api_key)
geocoders["google"] = GoogleV3(api_key=google_api_key)

In [ ]:
# geocoders["google"].geocode("Avenue Commandant de-Laage-de-Meux, 6852 Paliseul").raw

In [ ]:
arcgis = ArcGIS(username=arcgis_username, password=arcgis_password, referer="www.smalsresearch.Be")
arcgis._geocode = arcgis.geocode  # pylint: disable=protected-access
arcgis.geocode = lambda x: arcgis._geocode(x, out_fields="*")  # pylint: disable=protected-access, unexpected-keyword-arg
geocoders["arcgis"] = arcgis

# Load data


In [ ]:
data = pd.read_csv(f"{data_dir}/{dataset}.csv.gz", dtype=str)

mandatory_columns = ["street", "housenumber", "postcode", "city"]
for f in mandatory_columns:
    assert f in data, f"Field {f} is mandatory !!"

# data = data.sample(100, random_state=0).reset_index(drop=True)

In [ ]:
data["address"] = data.street.fillna("") + ", " + data.housenumber.fillna("") + ", " + data.postcode + " " + data.city + ", Belgique"
data

In [ ]:
data[~data.housenumber.fillna("").str.match("^[0-9]+$")]

In [ ]:
data = data.reset_index().rename(columns={"index": "reference_key"})

In [ ]:
pdf = PdfPages(f"output/geocoding/reports/report_{dataset}.pdf")

In [ ]:
metadata = [["Dataset", dataset],
            ["Nb records", data.shape[0]]]

fig, ax = plt.subplots()
ax.set_axis_off()
tbl = ax.table(cellText=[r[1:] for r in metadata],
               rowLabels=[r[0] for r in metadata],
               rowColours=["palegreen"] * len(metadata),
               loc="upper left")
tbl.scale(1, 2)
pdf_savefig()

# Geocode

In [ ]:
try:
    no_res_timing = pd.read_pickle(f"output/geocoding/no_res_timing_{dataset}.pkl")
except FileNotFoundError:
    no_res_timing = pd.DataFrame(columns=["address", "geocoder", "duration"])



In [ ]:

def geocode_and_timeit(geocoder, addr):
    """Geocode an address and time the operation. If no result, store the timing in a global DataFrame.

    Args:
        geocoder (str): name of the geocoder to use
        addr (str/dict): address to geocode

    Returns:
        pd.DataFrame: geocoded result or None
    """

    global no_res_timing  # pylint: disable=global-statement
    start = datetime.datetime.now()
    res = geocoder(addr)
    if res:
        res.raw.update({"duration": datetime.datetime.now() - start})
    else:
        geocoder_name = list(filter(lambda y: y[1] == geocoder.__self__, [(x, gx) for x, gx in geocoders.items()]))[0][0]

        no_res_timing = pd.concat([no_res_timing, pd.DataFrame([{"address":  addr,
                                                                 "duration": (datetime.datetime.now()-start).total_seconds(),
                                                                 "geocoder": geocoder_name}])])
    return res

In [ ]:


geocoded_data_r = []

coders = ["here", "bing", "google", "nominatim", "nominatim_wrapper", "be_pelias"]

from_file = []

# coders = ["here", "bing", "tomtom", "google", "nominatim", "pelias_unstruct", "pelias_struct", "pelias_struct_noloc", "be_pelias"]
# coders = ["ngi", "ngi_struct", "be_pelias", "arcgis"] #, "ngi_struct"]
coders = ["ngi", "ngi_struct", "be_pelias", "be_pelias_struct", "arcgis"]
# coders = ["be_pelias", "be_pelias_struct"]

coders = ["ngi_prd", "ngi_prd_struct",  "be_pelias", "be_pelias_struct", "arcgis", "here", "google"]
from_file = coders

delay = {"mapbox": 0.5, "nominatim_wrapper": 0.01,  "nominatim": 0.01, "lpost_bestaddress": 0.01, "google": 0.05, "tomtom": 0.4,
         "bing": 0.05, "arcgis": 0.05, "ngi": 0.01, "ngi_struct": 0.01, "ngi_prd": 0.02, "ngi_prd_struct": 0.02,
         "irisnet": 0.01, "irisnet_struct": 0.01}

for c in coders:
    if "pelias" in c:
        delay[c] = 0.01
    if "bestaddress" in c:
        delay[c] = 0.05

for coder in coders:
    print(coder)

    if coder in from_file:
        print("Load from local file...")
        g_data = pd.read_pickle(f"output/geocoding/geocoded_data/geocoded_{dataset}_{coder}.pkl")
    else:
        # Erase no-res timing for this coder for any previous run
        no_res_timing = no_res_timing[no_res_timing.geocoder != coder]

        g = geocoders[coder]
        g.geocode_timeit = lambda x: geocode_and_timeit(g.geocode, x)
        geocode = RateLimiter(g.geocode_timeit, min_delay_seconds=delay[coder] if coder in delay else 0.2)

        if coder.startswith("bestaddress") or "_struct" in coder:  # structured addresses
            print("  -> Structured geocoding")
            g_data = data.assign(location=data[["street", "housenumber", "postcode", "city"]].apply(dict, axis=1).progress_apply(geocode)).assign(geocoder=coder)

        else:
            print("  -> Unstructured geocoding")
            if coder == "tomtom":
                addr = data['address'].str.replace("/", "%2F")
            elif "ngi" in coder:
                addr = data['address'].str.replace(", Belgique", "")
            else:
                addr = data['address']
            g_data = data.assign(location=addr.progress_apply(geocode)).assign(geocoder=coder)
        g_data.to_pickle(f"output/geocoding/geocoded_data/geocoded_{dataset}_{coder}.pkl")

    geocoded_data_r.append(g_data)
    display(g_data[g_data.location.notnull()])

In [ ]:
# geocoders["irisnet"].verbose=True  # geocoders["ngi"].geocode(data.loc[830].to_dict())
geocoders["ngi_prd"].geocode(data.loc[5019].address.replace(", Belgique", ""), exactly_one=False)
# geocoders["irisnet"].geocode('Rue du Prince Albert, 1050 Ixelles')
# geocoders["ngi_prd"].verbose = True
# geocoders["ngi_prd"].geocode(data.loc[1191].address.replace(", Belgique", ""), exactly_one=False)
# geocoders["ngi_prd"].verbose = False

In [ ]:
data.loc[1192].street

## Gather results

In [ ]:
geocoded_data = pd.concat(geocoded_data_r).reset_index(drop=True)
geocoded_data = geocoded_data[geocoded_data.location.notnull()]
geocoded_data  # pylint: disable=pointless-statement

In [ ]:
geocoded_data[geocoded_data.location.astype(str).str.contains("GeocoderServiceError")][["location", "geocoder"]].astype(str).value_counts()

In [ ]:
geocoded_data[geocoded_data.location.astype(str).str.contains("GeocoderServiceError")]   # pylint: disable=expression-not-assigned

In [ ]:
for g in coders:
    data[~data.reference_key.isin(geocoded_data[geocoded_data.geocoder == g].reference_key.unique())].to_excel(f"{output_dir}/errors/no_res_{dataset}_{g}.xlsx", index=False)

In [ ]:
# if "be_pelias" in coders:
#     print("Mean Pelias call count: ", end="")
#     print(geocoded_data[geocoded_data.geocoder == "be_pelias"].location.apply(lambda r: r.raw["bepelias"]["peliasCallCount"]).mean())

# Duration

In [ ]:
geocoded_data["duration"] = geocoded_data["location"].apply(lambda loc: loc.raw["duration"] if loc and "duration" in loc.raw else None).dt.total_seconds()

In [ ]:
no_res_timing.to_pickle(f"output/geocoding/no_res_timing_{dataset}.pkl")

In [ ]:
geocoded_data_duration = pd.concat([geocoded_data.assign(with_result=True),
                                    no_res_timing[no_res_timing.geocoder.isin(coders)].assign(with_result=False)])
geocoded_data_duration = geocoded_data_duration.reset_index()[["address", "geocoder", "duration", "with_result"]]
geocoded_data_duration["address"] = geocoded_data_duration["address"].astype(str)

In [ ]:
plt.figure(figsize=def_figsize)
sbn.histplot(geocoded_data_duration[geocoded_data_duration.duration < geocoded_data_duration.duration.quantile(0.999)].sort_values("geocoder"),
             x="duration", hue="geocoder", element="poly",
             bins=100).set(title=f'Call duration distribution ({dataset})', xlabel="Duration (s)")

pdf_savefig()

In [ ]:
colors = [f"C{i}" for i in range(geocoded_data.geocoder.nunique())]

geocoded_data_duration.groupby("geocoder").duration.mean().plot.bar(color=colors,
                                                                    title=f"Geocoding duration ({dataset})",
                                                                    figsize=def_figsize)
plt.savefig(f'{fig_path}/geocoding_speed.png', dpi=150,  bbox_inches='tight')

pdf_savefig()

In [ ]:
geocoded_data_duration.groupby(["geocoder", "with_result"]).duration.mean().unstack().plot.bar(title=f"Geocoding duration ({dataset})",
                                                                                               figsize=def_figsize)

In [ ]:
colors = [f"C{i}" for i in range(geocoded_data.geocoder.nunique())]

(1 / geocoded_data_duration.groupby("geocoder").duration.mean()).plot.bar(color=colors,
                                                                          title=f"Geocoding rate (addr/sec - {dataset})",
                                                                          figsize=def_figsize)
# plt.savefig(f'{fig_path}/geocoding_speed_rate.png', dpi=150,  bbox_inches='tight')

pdf_savefig()

# Matching rate

## Matching rate

In [ ]:
mr = geocoded_data.groupby("geocoder").address.count() / data.shape[0]

In [ ]:


def show_values_on_hbars(ax, values):
    """ Add a second y-axis to show the values of a horizontal bar plot as percentages"""
    ax2 = ax.twinx()
    ax2.set_ylim(ax.get_ylim())
    ax2.set_yticks(ax.get_yticks())
    ax2.set_yticklabels(values.apply("{:.1%}".format))  # pylint: disable=consider-using-f-string
    # ax2.xaxis.set_major_formatter('')
    ax2.grid(axis='y')


plt.figure(figsize=def_figsize)
sbn.set(font_scale=1)

mr_plot = mr.rename("matching rate").plot.barh(title=f"Matching rate ({dataset})")

show_values_on_hbars(mr_plot, mr)

mr_plot.xaxis.set_major_formatter('{x:.0%}')

pdf_savefig()

sbn.set(font_scale=1)

## Extract precision

In [ ]:
geocoded_data[["precision", "precision_raw"]] = geocoded_data.apply(get_precision, axis=1).apply(pd.Series)

In [ ]:
geocoded_data[geocoded_data.precision.str.contains("UNK")][["geocoder", "precision"]].value_counts()

In [ ]:
geocoded_data[geocoded_data.precision.str.contains("UNK")]  # pylint: disable=expression-not-assigned

In [ ]:
geocoded_data[geocoded_data.precision.str.contains("UNK")].address

In [ ]:
# geocoded_data[geocoded_data.precision.str.contains("UNK")].iloc[0].location.raw
# geocoded_data[geocoded_data.precision_raw == "street_00"] 

In [ ]:
# geocoded_data[geocoded_data.precision == "country"]

In [ ]:
geocoded_data["precision"] = geocoded_data.precision.replace(r"^\[UNKNOWN.*\]", "[UNKNOWN]", regex=True)

## Precision

In [ ]:


def plot_matching_rate(geocoded_data, title=f"Matching rate - Precision ({dataset})"):
    """ Plot the matching rate by geocoder and precision level."""
    sbn.set(font_scale=1)
    mr_prec = geocoded_data.groupby(["geocoder", "precision"]).address.count().unstack()
    mr_prec = mr_prec / data.shape[0]
    mr_prec = mr_prec.reindex(["building", "street", "city", "country"], axis=1).reindex(coders, axis=0)

    mr_prec_plot = mr_prec.plot.barh(stacked=True,
                                     figsize=def_figsize,
                                     title=title,
                                     color=["tab:green", "tab:orange", "tab:red", "darkred", "tab:grey"])
    mr_prec_plot.xaxis.set_major_formatter('{x:.0%}')

    mr_prec_plot.set_ylabel(None)

    plt.legend(loc='lower left')

    show_values_on_hbars(mr_prec_plot, mr.reindex(coders))

    for bars in mr_prec_plot.containers:
        mr_prec_plot.bar_label(bars, label_type="center",
                               fmt=lambda x: f"{x:.0%}" if x > 0.005 else "",
                               fontproperties={"size": "x-small"})

    plt.savefig(f'{fig_path}/matching_rate_{dataset}.png', dpi=75,  bbox_inches='tight')

    sbn.set(font_scale=1)

    return mr_prec


In [ ]:


mr_prec = plot_matching_rate(geocoded_data)
pdf_savefig()

In [ ]:
with pd.option_context('display.float_format', '{:0.1%}'.format):  # pylint: disable=consider-using-f-string
    display(mr_prec.fillna(""))


In [ ]:


def has_boxinfo(f):
    """ Check if a geocoded result has box_info in its raw properties."""
    try:
        return "box_info" in f.raw["properties"]["addendum"]["best"]
    except KeyError:
        return False


In [ ]:


addr_in_bru = geocoded_data[geocoded_data.postcode.astype(int).between(1000, 1299)].address
geocoded_data_bru = geocoded_data[geocoded_data.address.isin(addr_in_bru)]

# mr_prec = plot_matching_rate(geocoded_data_bru, f"Matching rate - Precision (within Brussels - {dataset})")
# pdf_savefig()

In [ ]:
# if "be_pelias" in geocoded_data.geocoder.unique():
#     vc = geocoded_data[geocoded_data.geocoder == "be_pelias"].location.apply(lambda rec: rec.raw["bepelias"]["callType"]+" - "+(rec.raw["bepelias"]["transformers"] or "*")).value_counts()
#     display(vc)

#     _vc = vc.reset_index()
#     _vc.location = _vc.location.where(_vc["count"]>200, "other")
#     _vc.groupby("location")["count"].sum().sort_values(ascending=False).rename("").plot.pie(autopct='%1.1f%%',
#                                                                                  title=f"Transformer ({dataset})")

# #     vc.plot.pie()

# Distance to median statistics

## Extract location

In [ ]:
lat = geocoded_data["location"].apply(lambda loc: loc.point.latitude if loc and loc.point.latitude > 0 else None)
lng = geocoded_data["location"].apply(lambda loc: loc.point.longitude if loc else None)
geocoded_data["point"] = np.where(lat.notnull(), gpd.points_from_xy(lng, lat), pd.NA)

In [ ]:
geocoded_data[geocoded_data.point.isnull()]  # pylint: disable=expression-not-assigned

In [ ]:
# geocoded_data = geocoded_data[geocoded_data.point.notnull()]

In [ ]:
geocoded_data = gpd.GeoDataFrame(geocoded_data).set_geometry("point").set_crs(osm_crs, allow_override=True).to_crs(crs)
geocoded_data

In [ ]:
colors = ListedColormap([f"C{i}" for i in range(geocoded_data.geocoder.nunique())], name="mycmap")
ax = geocoded_data.plot("geocoder", legend=True, figsize=def_figsize, cmap=colors)
plt.title(f"Global view ({dataset})")
add_basemap(ax)
plt.axis("off")
pdf_savefig()


In [ ]:
belgium_boundaries = gpd.read_file("data/boundaries.geojson", driver="GeoJSON")
bru_boundaries = belgium_boundaries[belgium_boundaries.name == "BRU"].geometry.iloc[0].simplify(1000).buffer(1000)
belgium_boundaries = belgium_boundaries[belgium_boundaries.name == "BEL"].geometry.iloc[0].simplify(1000).buffer(1000)
# belgium_boundaries

In [ ]:
geocoded_data["is_in_belgium"] = geocoded_data.geometry.within(belgium_boundaries)
geocoded_data["is_in_belgium"].value_counts()

In [ ]:
geocoded_data[~geocoded_data.is_in_belgium & geocoded_data.point.notnull()]  # pylint: disable=expression-not-assigned

In [ ]:
geocoded_data.groupby("geocoder").is_in_belgium.value_counts()

In [ ]:
ax = geocoded_data[geocoded_data.is_in_belgium].plot("geocoder", legend=True, figsize=def_figsize)
plt.title("Belgian view")
add_basemap(ax)
pdf_savefig()



In [ ]:
#  geocoded_data[geocoded_data.is_in_belgium].drop(columns=["location"]).explore("geocoder")

## Compute (distance to) median

In [ ]:


def get_median_point(bloc):
    """ Get the median point of a GeoSeries of points. Return None if empty."""
    bloc = gpd.GeoSeries(bloc.dropna())
    if bloc.empty:
        return None, 0
    x = np.median(bloc.x)
    y = np.median(bloc.y)
    # display(bloc)
    # print(x, y)
    return shapely.geometry.Point(x, y), bloc.shape[0]



In [ ]:

median_geocoders = ["here", "bing", "mapbox", "tomtom", "nominatim_wrapper", "google", "be_pelias_struct", "arcgis", "ngi_struct"]  # Median based on those geocoders
median_points = geocoded_data[geocoded_data.geocoder.isin(median_geocoders) & (geocoded_data.precision == "building")].groupby("address").point.apply(get_median_point).apply(pd.Series)
median_points = median_points.rename(columns={0: "median_point", 1: "nb_points"}).reset_index()
median_points = median_points.set_geometry("median_point").set_crs(geocoded_data.crs)
median_points  # pylint: disable=pointless-statement

In [ ]:
median_points.nb_points.value_counts().plot.pie()

In [ ]:
median_points = median_points[median_points.nb_points >= min(3, median_points.nb_points.max())]
median_points.shape

In [ ]:
geocoded_data = geocoded_data.merge(median_points, how="left")

In [ ]:
geocoded_data["dist_to_median"] = geocoded_data.point.distance(geocoded_data.median_point.set_crs(geocoded_data.crs))
geocoded_data.sort_values("dist_to_median")


## Outsiders

In [ ]:


def plot_address(geocoded_data, addr=None, ref_key=None):
    """ Plot the geocoded results for a given address or reference key."""
    if ref_key:
        df = geocoded_data[(geocoded_data.reference_key == ref_key)]
        addr = df.address.iloc[0]
    else:
        df = geocoded_data[(geocoded_data.address == addr)]

    display(df[["reference_key", "street", "housenumber", "postcode", "city", "address"]].drop_duplicates())
    display(df.drop(columns=["reference_key", "street", "housenumber", "postcode", "city", "address"]))

    ax = df.assign(label=df.geocoder + " ("+df.dist_to_median.fillna(-1).astype(int).astype(str) + " - " + df.precision + ")").plot(
        "label",
        legend=True,
        markersize=100,
        figsize=def_figsize
    )

    if df.dist_to_median.isnull().all():
        print("No median point")
    else:
        med = df[["median_point"]].drop_duplicates().set_geometry("median_point")
        med.plot(ax=ax, color="red", marker="x", markersize=100)
    ax.set_title(addr)
    set_optimal_limits(ax, df)
    add_basemap(ax)
    plt.show()
    return ax



In [ ]:

plot_address(geocoded_data, ref_key=23)

In [ ]:
# for a in geocoded_data.sort_values("dist_to_median", ascending=False).address.unique()[0:5]:
#     print(a)
#     plot_address(geocoded_data, a)

In [ ]:
# geocoded_data[geocoded_data.dist_to_median.notnull() & geocoded_data.geocoder.str.startswith("be_pelias")].sort_values("dist_to_median", ascending=False).iloc[0:60]

In [ ]:
# plot_address(geocoded_data, ref_key=926)

In [ ]:
# geocoders["ngi_struct"].geocode(data.loc[34].address, exactly_one=False)
# geocoders["be_pelias_struct"].verbose=True
# geocoders["be_pelias_struct"].geocode(data.loc[5709].to_dict(), exactly_one=False)
# geocoders["be_pelias_struct"].geocode(data.loc[5302].address, exactly_one=False)

In [ ]:
geocoded_data[geocoded_data.dist_to_median == 0].geocoder.value_counts()

In [ ]:
geocoded_data.geocoder.value_counts()

## Median pertinence

How many close to median ?

In [ ]:
nb_close_to_median = geocoded_data.assign(close_to_median=geocoded_data.dist_to_median < 100).groupby("address").close_to_median.sum()
nb_close_to_median  # pylint: disable=pointless-statement

In [ ]:
median_points = median_points.merge(nb_close_to_median.rename("nb_close_to_median").reset_index())

In [ ]:
median_points.nb_close_to_median.value_counts()

In [ ]:
median_points[median_points.nb_close_to_median >= 2]

In [ ]:
# nb_close_to_median.value_counts()

In [ ]:
geocoded_data = geocoded_data.merge(median_points, how="left")

In [ ]:
geocoded_data.loc[geocoded_data.nb_close_to_median < 2, "dist_to_median"] = pd.NA

In [ ]:
# geocoded_data[geocoded_data.dist_to_median.notnull()].address.nunique()
# geocoded_data[geocoded_data.point.isnull()]

In [ ]:
for a in nb_close_to_median[nb_close_to_median == 0].iloc[0:4].index:
    if geocoded_data[geocoded_data.address == a].median_point.notnull().any():
        plot_address(geocoded_data[geocoded_data.median_point.notnull()], a)

In [ ]:
geocoded_data.sort_values("dist_to_median", ascending=False)

In [ ]:
geocoded_data.to_pickle(f"output/geocoding/geocoded_data/geocoded_{dataset}_all.pkl")

## Distance to median

### Average

In [ ]:
# d2m = geocoded_data.groupby("geocoder").dist_to_median.mean()
d2m = geocoded_data.groupby("geocoder").dist_to_median.mean()
d2m.plot.bar()
# plt.savefig(f"{fig_path}/dist2med_{dataset}.png")
d2m

In [ ]:
# skipping non reliable median

# d2m = geocoded_data[geocoded_data.is_reliable].groupby("geocoder").dist_to_median.mean()
d2m = geocoded_data[geocoded_data.is_in_belgium].groupby("geocoder").dist_to_median.mean()
d2m.plot.bar()
# plt.savefig(f"{fig_path}/dist2med_reliable_{dataset}.png")
d2m

In [ ]:
thresh = 100
ax = (geocoded_data[geocoded_data.dist_to_median < thresh].geocoder.value_counts() / data.shape[0]).plot.bar(title=f"Ratio of result closer than {thresh / 1000} km from median")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1%}", (p.get_x() * 1.005, p.get_height() * 1.005))

In [ ]:
# geocoded_data[geocoded_data.dist_to_median<thresh][["geocoder", "precision"]].value_counts()
# geocoded_data[geocoded_data.dist_to_median<thresh].groupby("geocoder").precision.value_counts().unstack().plot.bar(stacked=True)

In [ ]:
thresh = 1000
ax = (geocoded_data[geocoded_data.dist_to_median < thresh].geocoder.value_counts() / data.shape[0]).plot.bar(title=f"Ratio of result closer than {thresh / 1000} km from median")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1%}", (p.get_x() * 1.005, p.get_height() * 1.005))

In [ ]:
geocoded_data.assign(lower_than_thresh=geocoded_data.dist_to_median < thresh).groupby("geocoder").lower_than_thresh.value_counts().unstack().sort_index(ascending=False, axis=1).plot.bar(stacked=True)

### Percentiles

In [ ]:
# [geocoded_data.precision == "building"]
# from matplotlib.ticker import FormatStrFormatter


def dist_to_median_line_plot(d, ax=None, title=None, legend=True):
    """ Plot the distance to median for each geocoder as a line plot with mean, median, 95th percentile and max values."""

    bnd = 1
    d = d.assign(dst_to_med=d.dist_to_median.where(d.dist_to_median > bnd, bnd))

    # display(d)
    mean = d.groupby("geocoder").dist_to_median.mean().rename("Mean")

    # perc_90 = d.groupby("geocoder").dst_to_med.quantile(0.90).rename("Perc 90%")
    perc_95 = d.groupby("geocoder").dst_to_med.quantile(0.95).rename("Perc 95%")
    # perc_99 = d.groupby("geocoder").dst_to_med.quantile(0.99).rename("Perc 99%")
    perc_50 = d.groupby("geocoder").dst_to_med.quantile(0.50).rename("Median")
    maxd = d.groupby("geocoder").dst_to_med.max().rename("Max")
    tops = [d.groupby("geocoder").dst_to_med.nlargest(i).reset_index().groupby("geocoder").dst_to_med.min() for i in range(2, 50)]
#     mean    = mean.sort_values()
#     perc_90 = perc_90.reindex_like(mean)
#     perc_99 = perc_99.reindex_like(mean)
#     perc_50 = perc_50.reindex_like(mean)
#     maxd =    maxd.reindex_like(mean)

    mean = mean.reindex(coders)
#     perc_90 = perc_90.reindex(coders)
    perc_95 = perc_95.reindex(coders)
    perc_50 = perc_50.reindex(coders)
    maxd = maxd.reindex(coders)
    tops = [t.reindex(coders) for t in tops]

    display(mean)
    ax = mean.plot(legend=legend, linestyle='dotted', marker='s', ax=ax, title=title, figsize=def_figsize)
    ax = perc_50.plot(ax=ax, legend=legend, linestyle='dotted', marker='<')
    ax = perc_95.plot(ax=ax, legend=legend, linestyle='dotted', marker='P')
    ax = maxd.plot(ax=ax, rot=90, legend=legend, linestyle='dotted', marker='^', logy=True)
    plt.legend(bbox_to_anchor=(1.0, 1.0))
    for t in tops:
        t.plot(ax=ax, rot=90, legend=False, linestyle="", marker='.', logy=True, c='C3', zorder=1, alpha=0.2)
    ax.set_ylabel(f"Distance to median point ({dataset})")
    ax.tick_params(axis="both", direction="out", which="both", left=True, top=False)
    ax.set_xlabel(None)
    display(pd.DataFrame([mean, perc_50, perc_95, maxd]))



In [ ]:

dist_to_median_line_plot(geocoded_data)

pdf_savefig()


In [ ]:
# f, ax=plt.subplots(1, 2, figsize=(12, 5), sharey=True)
# dist_to_median_line_plot(geocoded_data, ax[0], "All results")

# # Keep only results containing input postcode (not fully accurate...)
# d = geocoded_data[geocoded_data.apply(lambda rec: str(rec.postcode) in  str(rec.location.raw), axis=1)]

# dist_to_median_line_plot(d, ax[1], "Excluding zipcode mismatches", legend=False)

# d.groupby("geocoder").reference_key.count()/ geocoded_data.groupby("geocoder").reference_key.count()

In [ ]:
# f, ax=plt.subplots(1, 2, figsize=(12, 5), sharey=True)
# dist_to_median_line_plot(geocoded_data, ax[0], "All results")

# # Keep only results containing input postcode (not fully accurate...)
# d = geocoded_data[geocoded_data.is_in_belgium]

# dist_to_median_line_plot(d, ax[1], "Excluding outside Belgium", legend=False)

# d.groupby("geocoder").reference_key.count()/ geocoded_data.groupby("geocoder").reference_key.count()

In [ ]:
# f, ax=plt.subplots(1, 2, figsize=(12, 5), sharey=True)
# dist_to_median_line_plot(geocoded_data, ax[0], "All results")

# # Keep only results containing input postcode (not fully accurate...)
# d = geocoded_data[geocoded_data.precision.isin(["building", 'street'])]

# dist_to_median_line_plot(d, ax[1], "Only 'building', 'street'")

# pdf_savefig()


In [ ]:
f, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
dist_to_median_line_plot(geocoded_data, ax[0], "All results")

# Keep only results containing input postcode (not fully accurate...)
d = geocoded_data[geocoded_data.precision.isin(["building", 'street']) & geocoded_data.apply(lambda rec: str(rec.postcode) in str(rec.location.raw), axis=1)]

dist_to_median_line_plot(d, ax[1], "Excluding zipcode mismatches, \nonly 'building' and 'street'", legend=False)

d.groupby("geocoder").reference_key.count() / geocoded_data.groupby("geocoder").reference_key.count()

# Compare Two geocoders

In [ ]:
g1 = "be_pelias_struct"
# g1 = "ngi"

# g2 = "irisnet"
g2 = "ngi_prd_struct"
# g2 = "ngi_struct"
# g2 = "bestaddress_adv_chk"
# g2 = "be_pelias_new_struct"


def format_id(loc):
    """ Format the identifier of a geocoded result as a string."""
    return f"{loc['namespace']}/{loc['objectIdentifier']}/{loc['versionIdentifier']}" if loc and "namespace" in loc else None


gdata1 = geocoded_data[(geocoded_data.geocoder == g1)].copy()
gdata2 = geocoded_data[(geocoded_data.geocoder == g2)].copy()

for g, gdata in ((g1, gdata1), (g2, gdata2)):
    if "ngi" in g:
        gdata["streetId"] = gdata.location.apply(lambda loc: loc.raw.get("hasStreetName", {}).get("id", ""))
        street_level = gdata.location.apply(lambda loc:  loc.raw.get("precision") == "street")
        gdata["streetId"] = gdata["streetId"].where(~street_level, gdata.location.apply(lambda loc: loc.raw.get("identifier")).apply(format_id))

        gdata["streetId"] = gdata["streetId"].str.replace("straatnaam//", "straatnaam/")

    if "be_pelias" in g:
        gdata["streetId"] = gdata.location.apply(lambda loc: loc.raw.get("street", {}).get("id"))
    if "irisnet" in g:
        gdata["streetId"] = gdata.location.apply(lambda loc: format_id(loc.raw.get("address", {}).get("hasStreetName", {}).get("code", {})))

        gdata["streetId"] = gdata["streetId"].where(gdata["streetId"].notnull(), gdata.location.apply(lambda loc: loc.raw.get("href") if loc.raw.get("precision") == "street" else None))

In [ ]:
gdata1.shape, gdata2.shape

In [ ]:
# street_code["namespace"] for gdata2.iloc[0].location.raw["address"]["hasStreetName"]["code"] as street_code
# gdata2

# gdata2
# street_id_mismatch[street_id_mismatch.reference_key == 561].location_ngi.iloc[0].raw
# gdata2[gdata2.precision == "street"]

In [ ]:
common_fields = ["reference_key", "street", "housenumber", "postcode", "city", "address"]
compare_fields = ["point", "location", "precision", "precision_raw", "streetId"]
g_mg = gdata1[common_fields + compare_fields].merge(gdata2[common_fields + compare_fields], on=common_fields, suffixes=("_"+g1, "_"+g2), how="outer")
g_mg

In [ ]:
g_mg["dist"] = g_mg.progress_apply(lambda r: r[f"point_{g1}"].distance(r[f"point_{g2}"]) if r[f"point_{g1}"] and r[f"point_{g2}"] else pd.NA, axis=1) if not g_mg.empty else pd.Series(dtype=float)

In [ ]:
geocoders["be_pelias_new_struct"].geocode(data.loc[7182].to_dict(), exactly_one=False)

In [ ]:
# g_mg.loc[7182].location_be_pelias_struct.raw
g_mg[g_mg.reference_key == 5019]

In [ ]:
g_mg[g_mg.dist.notnull()].sort_values("dist", na_position="first").iloc[-60:]

In [ ]:
street_id_mismatch = g_mg[(g_mg[f"streetId_{g1}"] != g_mg[f"streetId_{g2}"]) & g_mg[f"location_{g1}"].notnull() & g_mg[f"location_{g2}"].notnull()]
street_id_mismatch.sort_values("dist", na_position="last", ascending=False).to_excel(f"{output_dir}/errors/street_id_mismatch_{dataset}_{g1}_{g2}.xlsx", index=False)
street_id_mismatch.sort_values("dist", na_position="last", ascending=False)

In [ ]:
x = street_id_mismatch[street_id_mismatch[f"precision_{g1}"] != "city"].sort_values("dist", na_position="last", ascending=False)
x[["reference_key", "address", f"location_{g1}", f"location_{g2}", f"precision_{g1}",  f"precision_{g2}", "dist"]]

In [ ]:
list(x.iloc[0].location_be_pelias_struct.raw["street"]["name"].values())[0], x.iloc[0].street
x["match_with_street"] = x.progress_apply(lambda r: list(r.location_be_pelias_struct.raw["street"]["name"].values())[0], axis=1) == x.street
x

In [ ]:
x[x.match_with_street & (x.precision_ngi_prd_struct == "building") & (x.precision_be_pelias_struct == "building")][["reference_key", "address", f"location_{g1}", f"location_{g2}", f"precision_{g1}",  f"precision_{g2}", "dist"]]

In [ ]:
x[x.match_with_street][["reference_key", "address", f"location_{g1}", f"location_{g2}", f"precision_{g1}",  f"precision_{g2}", "dist"]]

In [ ]:
key = 121
geocoders[g2].verbose = True
display(data.loc[key])
res = geocoders[g2].geocode(data.loc[key].to_dict(), exactly_one=False)
geocoders[g2].verbose = False
res

In [ ]:
g_mg[g_mg[f"location_{g2}"].isnull()]

In [ ]:
# street_id_mismatch[street_id_mismatch.reference_key == 561]

In [ ]:
# geocoders["ngi"].verbose=True
# geocoders["ngi"].geocode("Merestraat, 175, 9300 Aalst, Belgique", exactly_one=False) #[0].raw
# geocoders["ngi"].geocode({"street": "Chaussée d'Arlon, NEU", "housenumber": "141", "postcode": "6840", "city": "Neufchâteau"}, exactly_one=False) #[0].raw

In [ ]:
# geocoders["be_pelias"].verbose=True
# geocoders["be_pelias"].geocode({"street": "Rue de Mont-Fat", "housenumber": "-", "postcode": "5500", "city": "Dinant"}, exactly_one=True).raw

In [ ]:
# gdata1.iloc[0].location.raw

In [ ]:
g_mg[g_mg.dist > 10].dist.hist(bins=50)

In [ ]:
exp = math.floor(math.log10(g_mg.dist.max())) + 1  # Get the smallest power of 10 greater than max distance
bins = [-0.01] + [10**i for i in range(0, exp+2, 2)]
bins = [-0.01] + [10**i for i in range(0, exp+1)]

In [ ]:
pd.cut(g_mg.dist.dropna(), bins).value_counts().sort_index().plot.pie()

In [ ]:
g_outliers = g_mg[g_mg.dist > 100].sort_values("dist")
g_outliers

In [ ]:
# geocoded_data[~geocoded_data.geometry.within(bru_boundaries) & geocoded_data.point.notnull()]

In [ ]:
ref_key = 658
plot_address(geocoded_data[geocoded_data.geocoder.isin([g1, g2])], ref_key=ref_key)
# data[data.reference_key == 810].address.iloc[0]

In [ ]:
# Input street not found in geocoding result
# g_mg[~g_mg.fillna("").apply(lambda r: r.street in str(r[f"location_{g2}"]), axis=1)]

In [ ]:
# Postcode mismatch
g_mg[~g_mg.apply(lambda r: str(r.postcode) in str(r[f"location_{g2}"]), axis=1)]

# Explore

In [ ]:
x = pd.DataFrame(geocoded_data[(geocoded_data.geocoder == "be_pelias")].sort_values("dist_to_median", na_position="last", ascending=False))
x.iloc[0:60]  # pylint: disable=pointless-statement

In [ ]:
# Non trouvable
# - Spaans-Huisdreef --> Ok, should not give postcode for streets, but nis
# housenumber: "157-159" ok, mais pas "157/159"
# - Droogveldstraat (https://best.pr.fedservices.be/api/opendata/best/v1/belgianAddress/v2/addresses?streetName=Droogveldstraat&postCode=3404&houseNumber=3).
#   Pas de resultat, mais le résultat existant est "proposed" (marche si on rajoute &status=proposed)
# - https://best.pr.fedservices.be/api/opendata/best/v1/belgianAddress/v2/addresses?streetName=Tiegemstraat&postCode=8553&houseNumber=22C:
#   trouvé avec "22C", pas avec "22 C". Idem pour "22A, Pannenhuisstraat, 2222, Heist-op-den-Berg"
# rue de l'abattoir: trouvé avec 7131, pas avec 7130
# - mismatch entre zipcode de bpost et Best. Pos
# - streetname avec ()


In [ ]:
ref_key = 140
display(data[data.reference_key == ref_key].iloc[[0]])
display(geocoded_data[geocoded_data.reference_key == ref_key])
geocoders["bestaddress_adv_chk"].verbose = True
geocoders["bestaddress_adv_chk"].geocode(data[data.reference_key == ref_key].iloc[0].to_dict())

In [ ]:
# geocoded_data[geocoded_data.reference_key == ref_key].drop(columns=["location"]).explore("geocoder", marker_kwds={"radius": 10})

In [ ]:
# geocoded_data.loc[66790].location.raw
# geocoded_data[(geocoded_data.reference_key == ref_key) & (geocoded_data.geocoder == "be_pelias")].iloc[0].location.raw

In [ ]:
idx = geocoded_data[(geocoded_data.reference_key == ref_key) & (geocoded_data.geocoder == "be_pelias")].index[0]

In [ ]:
geocoded_data.loc[idx]

In [ ]:
geocoders["bestaddress_adv_chk"].verbose = True
geocoders["bestaddress_adv_chk"].geocode(geocoded_data.loc[idx].to_dict()).raw

In [ ]:
geocoded_data

In [ ]:
r_struct = geocoded_data[geocoded_data["geocoder"] == "be_pelias"]
r_unstruct = geocoded_data[geocoded_data["geocoder"] == "be_pelias_unstruct"]

In [ ]:
mg = r_struct[["reference_key", "address", "location", "precision"]].merge(r_unstruct[["reference_key", "address", "location", "precision"]], on=["reference_key", "address"], how="outer")
mg

In [ ]:
mg_diff = mg[mg.location_x.apply(lambda x: x.address if not pd.isnull(x) else None) != mg.location_y.apply(lambda x: x.address if not pd.isnull(x) else None)]
mg_diff


In [ ]:
mg_diff[["precision_x", "precision_y"]].value_counts()

In [ ]:
ref_key = 200
geocoded_data[geocoded_data.reference_key == ref_key]

In [ ]:
idx = 200
geocoded_data.loc[idx]